In [81]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# Loading CSVs using path.cwd
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
MODELS = PROJECT_ROOT / "models"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"
ML_DATA = PROJECT_ROOT / "ml_data"
ML_DATA.mkdir(exist_ok=True)

model = joblib.load(MODELS / "lightgbm_tuned.pkl")
encoder = joblib.load(MODELS / "onehot_encoder.pkl")
feature_columns = joblib.load(MODELS / "feature_columns.pkl")


elo_df = pd.read_csv(PROCESSED_DATA / "elo_ratings_V3.csv")
team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")

In [82]:
print("Elo Ratings Shape:", elo_df.shape)
print("Team Matches Shape:", team_matches.shape)

Elo Ratings Shape: (210, 3)
Team Matches Shape: (47520, 8)


In [83]:
# Helper Function - Get Team Elo Rating

def get_elo(team_name):
    
    team = elo_df.loc[elo_df["team"] == team_name]
    
    if team.empty:
        raise ValueError(f"Team '{team_name}' not found in Elo ratings.")
    
    return float(team.iloc[0]["elo_rating"])

In [84]:
# Helper Function - Get Team Statistics


def get_team_stats(team_name):
    
    team_df = (
        team_matches[team_matches["team"] == team_name]
        .sort_values("date")
        .copy()
    )

    if team_df.empty:
        raise ValueError(f"No match history found for '{team_name}'.")

    # Overall statistics
    matches_played = len(team_df)
    wins = (team_df["result"] == "W").sum()

    win_pct = wins / matches_played

    avg_gf = team_df["goals_for"].mean()
    avg_ga = team_df["goals_against"].mean()

    # Last 5 matches
    # Last 5 matches
    last5 = team_df.tail(5)

    points = (
        (last5["result"] == "W").sum() * 3 +
        (last5["result"] == "D").sum()
    )

    last5_ppg = points / len(last5)

    last5_avg_gf = last5["goals_for"].mean()
    last5_avg_ga = last5["goals_against"].mean()

    return {
    "elo": float(get_elo(team_name)),
    "win_pct": float(win_pct),
    "avg_gf": float(avg_gf),
    "avg_ga": float(avg_ga),
    "last5_ppg": float(last5_ppg),
    "last5_avg_gf": float(last5_avg_gf),
    "last5_avg_ga": float(last5_avg_ga),
}

In [85]:
# Create Match Features

def create_match_features(home_team, away_team, neutral):
    """
    Creates the numerical feature vector for a match.
    """

    home = get_team_stats(home_team)
    away = get_team_stats(away_team)

    match_features = {
        "home_elo": home["elo"],
        "away_elo": away["elo"],
        "elo_diff": home["elo"] - away["elo"],

        "home_win_pct": home["win_pct"],
        "away_win_pct": away["win_pct"],

        "home_avg_gf": home["avg_gf"],
        "away_avg_gf": away["avg_gf"],

        "home_avg_ga": home["avg_ga"],
        "away_avg_ga": away["avg_ga"],

        "neutral": neutral,

        "home_last5_ppg": home["last5_ppg"],
        "away_last5_ppg": away["last5_ppg"],

        "home_last5_avg_gf": home["last5_avg_gf"],
        "away_last5_avg_gf": away["last5_avg_gf"],

        "home_last5_avg_ga": home["last5_avg_ga"],
        "away_last5_avg_ga": away["last5_avg_ga"],

        # Engineered Features
        "abs_elo_diff": abs(home["elo"] - away["elo"]),
        "win_pct_gap": home["win_pct"] - away["win_pct"],
        "last5_ppg_gap": home["last5_ppg"] - away["last5_ppg"],
        "avg_gf_gap": home["avg_gf"] - away["avg_gf"],
        "avg_ga_gap": home["avg_ga"] - away["avg_ga"],
    }

    return pd.DataFrame([match_features])

In [86]:
# Encode Tournament

def encode_tournament(match_df, tournament):
    """
    One-hot encodes the tournament and appends it to the
    numerical match features.
    """

    # Tournament input
    tournament_input = pd.DataFrame({
        "tournament": [tournament]
    })

    # Encode tournament
    tournament_encoded = encoder.transform(tournament_input)

    # Convert to DataFrame
    tournament_df = pd.DataFrame(
        tournament_encoded,
        columns=encoder.get_feature_names_out(["tournament"])
    ).astype(int)

    # Combine
    match_df = pd.concat(
        [match_df.reset_index(drop=True),
         tournament_df.reset_index(drop=True)],
        axis=1
    )

    return match_df

In [87]:
# Predict Match

def predict_match(home_team, away_team, tournament, neutral):
    """
    Predicts the outcome of a football match using the trained LightGBM model.
    """

    # Create numerical features
    match_df = create_match_features(
        home_team,
        away_team,
        neutral
    )

    # Encode tournament
    match_df = encode_tournament(
        match_df,
        tournament
    )

    # Align columns
    match_df = match_df.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # Prediction
    prediction = model.predict(match_df)[0]
    probabilities = model.predict_proba(match_df)[0]

    class_mapping = {
        0: "Away Win",
        1: "Draw",
        2: "Home Win"
    }

    print("=" * 50)
    print("        FIFA WORLD CUP PREDICTOR")
    print("=" * 50)

    print(f"Home Team : {home_team}")
    print(f"Away Team : {away_team}")
    #print(f"Tournament: {tournament}")
    print(f"Neutral   : {'Yes' if neutral else 'No'}")

    print("\nPredicted Result")
    print("-" * 20)
    print(class_mapping[prediction])

    print("\nPrediction Probabilities")
    print("-" * 30)
    print(f"Away Win : {probabilities[0]:.2%}")
    print(f"Draw     : {probabilities[1]:.2%}")
    print(f"Home Win : {probabilities[2]:.2%}")

    return prediction, probabilities

# Quarter-final Fixtures

The quarter-final fixtures are based on the official FIFA World Cup 2026 knockout stage.

These fixtures will be simulated using the trained LightGBM prediction model.

In [88]:
# ==========================================================
# 1. Quarter-final Fixtures
# ==========================================================

quarter_finals = [
    ("France", "Morocco"),
    ("Spain", "Belgium"),
    ("Norway", "England"),
    ("Argentina", "Switzerland")
]

tournament = "FIFA World Cup"
neutral = 1

print("Quarter-final Fixtures\n")

for i, (home, away) in enumerate(quarter_finals, start=1):
    print(f"QF{i}: {home} vs {away}")

Quarter-final Fixtures

QF1: France vs Morocco
QF2: Spain vs Belgium
QF3: Norway vs England
QF4: Argentina vs Switzerland


In [89]:
# 2. Simulate Knockout Round

def simulate_round(fixtures, round_name):
    """
    Simulates one knockout round.
    Returns winners and losers.
    """

    winners = []
    losers = []

    print("=" * 60)
    print(round_name.upper())
    print("=" * 60)

    for home, away in fixtures:

        prediction, probabilities = predict_match(
            home,
            away,
            tournament,
            neutral
        )

        # Home Win
        if prediction == 2:
            winner = home
            loser = away

        # Away Win
        elif prediction == 0:
            winner = away
            loser = home

        # Draw → choose higher win probability
        else:
            home_prob = probabilities[2]
            away_prob = probabilities[0]

            if home_prob >= away_prob:
                winner = home
                loser = away
            else:
                winner = away
                loser = home

        print(f"{home} vs {away}  -->  {winner}")

        winners.append(winner)
        losers.append(loser)

    return winners, losers

#def simulate_round(fixtures, round_name):
#    """
#    Simulates one knockout round and returns the winners.
#    """
#
#    winners = []
#
#    print("=" * 60)
#    print(round_name.upper())
#    print("=" * 60)
#
#    for home, away in fixtures:
#
#        prediction, probabilities = predict_match(
#            home,
#            away,
#            tournament,
#            neutral
#        )
#
#        # Home Win
#        if prediction == 2:
#            winner = home
#            loser = away
#
#        # Away Win
#        elif prediction == 0:
#            winner = away
#            loser = home
#
#        # Draw → choose team with higher win probability
#        else:
#            home_prob = probabilities[2]
#            away_prob = probabilities[0]
#            if home_prob >= away_prob:
#                winner = home
#                loser = away
#            else:
#                winner = away
#                loser = home
#
#        print(f"{home} vs {away}  -->  {winner}")
#
#        winner.append(winner)
#        loser.append(loser)
#
#    return winner, loser

# Simulate Knockout Round

This function simulates every match in a knockout round using the trained LightGBM model and returns the winning teams, which advance to the next round.

Since knockout matches cannot end in a draw, any predicted draw is resolved by selecting the team with the higher win probability.

In [90]:
quarter_final_winners, quarter_final_losers = simulate_round(
    quarter_finals,
    "Quarter-finals"
)

print("\nQualified for Semi-finals:")
print(quarter_final_winners)

QUARTER-FINALS
        FIFA WORLD CUP PREDICTOR
Home Team : France
Away Team : Morocco
Neutral   : Yes

Predicted Result
--------------------
Home Win

Prediction Probabilities
------------------------------
Away Win : 21.77%
Draw     : 21.10%
Home Win : 57.13%
France vs Morocco  -->  France
        FIFA WORLD CUP PREDICTOR
Home Team : Spain
Away Team : Belgium
Neutral   : Yes

Predicted Result
--------------------
Home Win

Prediction Probabilities
------------------------------
Away Win : 26.56%
Draw     : 25.11%
Home Win : 48.33%
Spain vs Belgium  -->  Spain
        FIFA WORLD CUP PREDICTOR
Home Team : Norway
Away Team : England
Neutral   : Yes

Predicted Result
--------------------
Away Win

Prediction Probabilities
------------------------------
Away Win : 54.95%
Draw     : 26.14%
Home Win : 18.91%
Norway vs England  -->  England
        FIFA WORLD CUP PREDICTOR
Home Team : Argentina
Away Team : Switzerland
Neutral   : Yes

Predicted Result
--------------------
Home Win

Predictio

# Semi-final Fixtures

The semi-final fixtures are automatically generated from the winners of the quarter-finals.

In [91]:
# ==========================================================
# 3. Semi-final Fixtures
# ==========================================================

semi_finals = [
    (quarter_final_winners[0], quarter_final_winners[1]),
    (quarter_final_winners[2], quarter_final_winners[3])
]

print("Semi-final Fixtures\n")

for i, (home, away) in enumerate(semi_finals, start=1):
    print(f"SF{i}: {home} vs {away}")

Semi-final Fixtures

SF1: France vs Spain
SF2: England vs Argentina


In [92]:
semi_final_winners, semi_final_losers = simulate_round(
    semi_finals,
    "Semi-finals"
)

print("\nQualified for Final:")
print(semi_final_winners)

SEMI-FINALS
        FIFA WORLD CUP PREDICTOR
Home Team : France
Away Team : Spain
Neutral   : Yes

Predicted Result
--------------------
Home Win

Prediction Probabilities
------------------------------
Away Win : 33.37%
Draw     : 31.31%
Home Win : 35.32%
France vs Spain  -->  France
        FIFA WORLD CUP PREDICTOR
Home Team : England
Away Team : Argentina
Neutral   : Yes

Predicted Result
--------------------
Away Win

Prediction Probabilities
------------------------------
Away Win : 46.70%
Draw     : 23.89%
Home Win : 29.41%
England vs Argentina  -->  Argentina

Qualified for Final:
['France', 'Argentina']


In [ ]:
# Third Place Playoff

third_place = [
    (semi_final_losers[0], semi_final_losers[1])
]

print("\nThird Place Playoff\n")
print(f"{third_place[0][0]} vs {third_place[0][1]}")

third_place_winner, _ = simulate_round(
    third_place,
    "Third Place Playoff"
)

print("\n🥉 THIRD PLACE")
print("=" * 30)
print(third_place_winner[0])


Third Place Playoff

Spain vs England
THIRD PLACE PLAYOFF
        FIFA WORLD CUP PREDICTOR
Home Team : Spain
Away Team : England
Neutral   : Yes

Predicted Result
--------------------
Away Win

Prediction Probabilities
------------------------------
Away Win : 37.98%
Draw     : 31.17%
Home Win : 30.84%
Spain vs England  -->  England
🥉 THIRD PLACE
England


# Final

The final is simulated using the winners of the two semi-finals to determine the predicted FIFA World Cup champion.

In [99]:
# 4. Final

final = [
    (semi_final_winners[0], semi_final_winners[1])
]

print("Final:")

print(f"{final[0][0]} VS {final[0][1]}")

Final:
France VS Argentina


In [100]:
final_winner = simulate_round(
    final,
    "Final"
)

champion = final_winner[0]

print("\n🏆 FIFA WORLD CUP 2026 CHAMPION 🏆")
print("=" * 40)
print(f"{champion} have won the FIFA World Cup 2026!")

FINAL
        FIFA WORLD CUP PREDICTOR
Home Team : France
Away Team : Argentina
Neutral   : Yes

Predicted Result
--------------------
Away Win

Prediction Probabilities
------------------------------
Away Win : 42.30%
Draw     : 25.00%
Home Win : 32.69%
France vs Argentina  -->  Argentina

🏆 FIFA WORLD CUP 2026 CHAMPION 🏆
['Argentina'] have won the FIFA World Cup 2026!
